# Synthetic Artifact Features

The competition data is synthetically generated.

Repeated and discretized numerical values may therefore contain
generator-specific predictive structure that ordinary tree models
do not fully exploit.

This experiment focuses first on Annual_Income_USD.

All target encoding is performed out-of-fold to avoid leakage.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.metrics import roc_auc_score

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

In [3]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

y = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

train.shape, test.shape

((668665, 15), (286571, 14))

In [4]:
print(
    "Rows:",
    len(train)
)

print(
    "Unique incomes:",
    train["Annual_Income_USD"].nunique()
)

print(
    "Average rows per income:",
    len(train)
    / train["Annual_Income_USD"].nunique()
)

Rows: 668665
Unique incomes: 13214
Average rows per income: 50.60276978961707


In [5]:
train[
    "Annual_Income_USD"
].value_counts().describe()

count    13214.000000
mean        50.602770
std        540.861771
min          1.000000
25%          1.000000
50%          8.000000
75%         66.000000
max      61605.000000
Name: count, dtype: float64

In [6]:
train[
    "Annual_Income_USD"
].value_counts().head(20)

Annual_Income_USD
30000.0     61605
86095.0      1082
96749.0      1073
65800.0      1029
92372.0       907
84578.0       905
93957.0       852
68480.0       762
82440.0       739
84768.0       725
80777.0       720
91412.0       712
123547.0      691
94863.0       687
74981.0       680
72441.0       674
91560.0       663
86844.0       659
72454.0       644
94575.0       627
Name: count, dtype: int64

In [7]:
def make_artifact_keys(df):
    keys = pd.DataFrame(
        index=df.index
    )

    income = df[
        "Annual_Income_USD"
    ]

    # Exact synthetic value
    keys["Income_Exact"] = (
        income
        .round(0)
        .astype("Int64")
        .astype(str)
    )

    # $50 bins
    keys["Income_Floor_50"] = (
        np.floor(income / 50) * 50
    ).astype("Int64").astype(str)

    # $250 bins
    keys["Income_Floor_250"] = (
        np.floor(income / 250) * 250
    ).astype("Int64").astype(str)

    # Slightly coarser comparison
    keys["Income_Floor_500"] = (
        np.floor(income / 500) * 500
    ).astype("Int64").astype(str)

    return keys

In [8]:
artifact_keys = make_artifact_keys(
    train
)

artifact_keys.head()

,Income_Exact,Income_Floor_50,Income_Floor_250,Income_Floor_500
0,92887,92850,92750,92500
1,30000,30000,30000,30000
2,94389,94350,94250,94000
3,73580,73550,73500,73500
4,57898,57850,57750,57500


In [9]:
artifact_keys.nunique()

Income_Exact        13214
Income_Floor_50      2026
Income_Floor_250      533
Income_Floor_500      284
dtype: int64

In [10]:
def build_smoothed_mapping(
    key,
    target,
    smoothing=50
):
    temp = pd.DataFrame({
        "key": key
            .astype(str)
            .to_numpy(),

        "target": target
            .to_numpy()
    })

    global_mean = target.mean()

    stats = (
        temp
        .groupby("key")["target"]
        .agg(["sum", "count"])
    )

    stats["encoding"] = (
        stats["sum"]
        + smoothing * global_mean
    ) / (
        stats["count"]
        + smoothing
    )

    return (
        stats["encoding"],
        global_mean
    )

In [11]:
def create_artifact_encodings(
    train_df,
    valid_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(
        drop=True
    )

    valid_df = valid_df.reset_index(
        drop=True
    )

    y_train = y_train.reset_index(
        drop=True
    )

    train_keys = make_artifact_keys(
        train_df
    )

    valid_keys = make_artifact_keys(
        valid_df
    )

    train_encoded = pd.DataFrame(
        index=train_df.index
    )

    valid_encoded = pd.DataFrame(
        index=valid_df.index
    )

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        oof_values = np.zeros(
            len(train_df)
        )

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):

            mapping, global_mean = (
                build_smoothed_mapping(
                    train_keys[col].iloc[
                        inner_train_idx
                    ],
                    y_train.iloc[
                        inner_train_idx
                    ],
                    smoothing=smoothing
                )
            )

            oof_values[
                inner_valid_idx
            ] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_encoded[
            f"TE_{col}"
        ] = oof_values

        full_mapping, full_global_mean = (
            build_smoothed_mapping(
                train_keys[col],
                y_train,
                smoothing=smoothing
            )
        )

        valid_encoded[
            f"TE_{col}"
        ] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

    return (
        train_encoded,
        valid_encoded
    )

In [12]:
def create_frequency_features(
    train_df,
    valid_df
):
    train_df = train_df.reset_index(
        drop=True
    )

    valid_df = valid_df.reset_index(
        drop=True
    )

    train_keys = make_artifact_keys(
        train_df
    )

    valid_keys = make_artifact_keys(
        valid_df
    )

    train_freq = pd.DataFrame(
        index=train_df.index
    )

    valid_freq = pd.DataFrame(
        index=valid_df.index
    )

    for col in train_keys.columns:

        counts = (
            train_keys[col]
            .value_counts()
        )

        train_freq[
            f"FREQ_{col}"
        ] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[
            f"FREQ_{col}"
        ] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_freq,
        valid_freq
    )

In [13]:
train_idx, valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

outer_train = train.iloc[
    train_idx
].copy()

outer_valid = train.iloc[
    valid_idx
].copy()

y_outer_train = y.iloc[
    train_idx
].copy()

y_outer_valid = y.iloc[
    valid_idx
].copy()

In [14]:
train_art_te, valid_art_te = (
    create_artifact_encodings(
        outer_train,
        outer_valid,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42
    )
)

In [15]:
train_art_freq, valid_art_freq = (
    create_frequency_features(
        outer_train,
        outer_valid
    )
)

In [16]:
train_art_te.head()

,TE_Income_Exact,TE_Income_Floor_50,TE_Income_Floor_250,TE_Income_Floor_500
0,0.125568,0.118716,0.100977,0.099945
1,0.237309,0.241288,0.181278,0.245190
2,0.186925,0.221954,0.221018,0.213112
3,0.265811,0.245730,0.230612,0.245034
4,0.193712,0.146562,0.146682,0.192291


In [17]:
train_art_freq.head()

,FREQ_Income_Exact,FREQ_Income_Floor_50,FREQ_Income_Floor_250,FREQ_Income_Floor_500
0,161,600,3537,3708
1,141,305,1426,2043
2,395,1192,1261,2148
3,129,457,1435,4607
4,110,906,922,5538


In [19]:
def add_features(df):
    df = df.copy()

    # Numerical interactions
    df["Income_per_Car"] = (
        df["Annual_Income_USD"]
        / df["Number_of_Cars_Owned"].clip(lower=1)
    )

    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"]
        + df["Charging_Stations_Near_Work"]
    )

    df["Charging_Station_Difference"] = (
        df["Charging_Stations_Near_Home"]
        - df["Charging_Stations_Near_Work"]
    )

    df["Income_x_Concern"] = (
        df["Annual_Income_USD"]
        * df["Environmental_Concern_Level"]
    )

    df["Commute_x_Concern"] = (
        df["Daily_Commute_km"]
        * df["Environmental_Concern_Level"]
    )

    # Categorical interactions
    df["Subsidy_x_Concern"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    df["Subsidy_x_Anxiety"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Concern_x_Anxiety"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    income_band = pd.cut(
        df["Annual_Income_USD"],
        bins=[
            -np.inf,
            60000,
            80000,
            100000,
            120000,
            np.inf
        ],
        labels=[
            "Income_1",
            "Income_2",
            "Income_3",
            "Income_4",
            "Income_5"
        ]
    ).astype(str)

    df["Income_x_Subsidy"] = (
        income_band
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["HomeCharge_x_Subsidy"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    commute_band = pd.cut(
        df["Daily_Commute_km"],
        bins=[
            -np.inf,
            20,
            40,
            60,
            np.inf
        ],
        labels=[
            "Short",
            "Medium",
            "Long",
            "VeryLong"
        ]
    ).astype(str)

    df["HomeCharge_x_Commute"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + commute_band
    )

    return df

In [20]:
train_fe = add_features(
    outer_train
)

valid_fe = add_features(
    outer_valid
)

In [21]:
X_train_art = (
    train_fe
    .drop(
        columns=[
            "id",
            "Will_Buy_EV"
        ]
    )
    .reset_index(drop=True)
)

In [23]:
X_train_art = (
    train_fe
    .drop(
        columns=[
            "id",
            "Will_Buy_EV"
        ]
    )
    .reset_index(drop=True)
)

X_valid_art = (
    valid_fe
    .drop(
        columns=[
            "id",
            "Will_Buy_EV"
        ]
    )
    .reset_index(drop=True)
)

X_train_art.shape, X_valid_art.shape

((534932, 24), (133733, 24))

In [24]:
X_train_art = pd.concat(
    [
        X_train_art,
        train_art_te,
        train_art_freq
    ],
    axis=1
)

X_valid_art = pd.concat(
    [
        X_valid_art,
        valid_art_te,
        valid_art_freq
    ],
    axis=1
)

In [25]:
categorical_cols = (
    X_train_art
    .select_dtypes(
        include=[
            "object",
            "str",
            "category"
        ]
    )
    .columns
    .tolist()
)

for col in categorical_cols:

    categories = pd.concat(
        [
            X_train_art[col].astype(str),
            X_valid_art[col].astype(str)
        ],
        ignore_index=True
    ).unique().tolist()

    X_train_art[col] = pd.Categorical(
        X_train_art[col].astype(str),
        categories=categories
    )

    X_valid_art[col] = pd.Categorical(
        X_valid_art[col].astype(str),
        categories=categories
    )

In [26]:
artifact_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [27]:
artifact_model.fit(
    X_train_art,
    y_outer_train.reset_index(
        drop=True
    ),

    eval_X=X_valid_art,
    eval_y=y_outer_valid.reset_index(
        drop=True
    ),

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.944814	valid_0's binary_logloss: 0.220955
[400]	valid_0's auc: 0.945165	valid_0's binary_logloss: 0.220189
[600]	valid_0's auc: 0.94525	valid_0's binary_logloss: 0.220006
[800]	valid_0's auc: 0.945254	valid_0's binary_logloss: 0.220003
Early stopping, best iteration is:
[847]	valid_0's auc: 0.945266	valid_0's binary_logloss: 0.219976
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [28]:
artifact_valid_pred = (
    artifact_model
    .predict_proba(
        X_valid_art
    )[:, 1]
)

artifact_auc = roc_auc_score(
    y_outer_valid,
    artifact_valid_pred
)

In [29]:
print(
    f"Raw LightGBM:           "
    f"{0.941669:.6f}"
)

print(
    f"Feature Engineering:    "
    f"{0.941785:.6f}"
)

print(
    f"FE + Target Encoding:   "
    f"{0.941771:.6f}"
)

print(
    f"FE + Original Rows:     "
    f"{0.941698:.6f}"
)

print(
    f"FE + Source Score:      "
    f"{0.941722:.6f}"
)

print(
    f"FE + Income Artifacts:  "
    f"{artifact_auc:.6f}"
)

print()

print(
    f"Gain vs FE: "
    f"{artifact_auc - 0.941785:+.6f}"
)

print(
    f"Best iteration: "
    f"{artifact_model.best_iteration_}"
)

Raw LightGBM:           0.941669
Feature Engineering:    0.941785
FE + Target Encoding:   0.941771
FE + Original Rows:     0.941698
FE + Source Score:      0.941722
FE + Income Artifacts:  0.945266

Gain vs FE: +0.003481
Best iteration: 847


In [30]:
artifact_importance = pd.DataFrame({
    "Feature":
        X_train_art.columns,

    "Importance":
        artifact_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

artifact_importance[
    artifact_importance[
        "Feature"
    ].str.contains(
        "TE_|FREQ_",
        regex=True
    )
]

,Feature,Importance
24,TE_Income_Exact,2482
25,TE_Income_Floor_50,1918
28,FREQ_Income_Exact,1758
29,FREQ_Income_Floor_50,1239
27,TE_Income_Floor_500,1219
26,TE_Income_Floor_250,1125
31,FREQ_Income_Floor_500,865
30,FREQ_Income_Floor_250,858


In [31]:
def create_artifact_features_three_way(
    train_df,
    valid_df,
    test_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    train_keys = make_artifact_keys(train_df)
    valid_keys = make_artifact_keys(valid_df)
    test_keys = make_artifact_keys(test_df)

    train_te = pd.DataFrame(index=train_df.index)
    valid_te = pd.DataFrame(index=valid_df.index)
    test_te = pd.DataFrame(index=test_df.index)

    train_freq = pd.DataFrame(index=train_df.index)
    valid_freq = pd.DataFrame(index=valid_df.index)
    test_freq = pd.DataFrame(index=test_df.index)

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        # -------------------------
        # Leak-free OOF TE
        # -------------------------
        oof_values = np.zeros(len(train_df))

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):
            mapping, global_mean = build_smoothed_mapping(
                train_keys[col].iloc[inner_train_idx],
                y_train.iloc[inner_train_idx],
                smoothing=smoothing
            )

            oof_values[inner_valid_idx] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_te[f"TE_{col}"] = oof_values

        # Mapping learned from whole OUTER training fold only
        full_mapping, full_global_mean = build_smoothed_mapping(
            train_keys[col],
            y_train,
            smoothing=smoothing
        )

        valid_te[f"TE_{col}"] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        test_te[f"TE_{col}"] = (
            test_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        # -------------------------
        # Frequency features
        # -------------------------
        counts = train_keys[col].value_counts()

        train_freq[f"FREQ_{col}"] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[f"FREQ_{col}"] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        test_freq[f"FREQ_{col}"] = (
            test_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    )

In [32]:
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions_artifact = np.zeros(len(train))
test_predictions_artifact = np.zeros(len(test))

fold_scores_artifact = []
best_iterations_artifact = []

In [33]:
for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(train, y),
    start=1
):

    print(f"\n{'=' * 60}")
    print(f"OUTER FOLD {fold}")
    print(f"{'=' * 60}")

    outer_train = train.iloc[train_idx].copy()
    outer_valid = train.iloc[valid_idx].copy()

    y_outer_train = y.iloc[train_idx].copy()
    y_outer_valid = y.iloc[valid_idx].copy()

    # --------------------------------
    # Leak-free synthetic artifacts
    # --------------------------------
    (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    ) = create_artifact_features_three_way(
        outer_train,
        outer_valid,
        test,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42 + fold
    )

    # --------------------------------
    # Ordinary engineered features
    # --------------------------------
    train_fe_fold = add_features(outer_train)
    valid_fe_fold = add_features(outer_valid)
    test_fe_fold = add_features(test)

    X_train_fold = (
        train_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_valid_fold = (
        valid_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_test_fold = (
        test_fe_fold
        .drop(columns=["id"])
        .reset_index(drop=True)
    )

    # --------------------------------
    # Append artifact features
    # --------------------------------
    X_train_fold = pd.concat(
        [
            X_train_fold,
            train_te,
            train_freq
        ],
        axis=1
    )

    X_valid_fold = pd.concat(
        [
            X_valid_fold,
            valid_te,
            valid_freq
        ],
        axis=1
    )

    X_test_fold = pd.concat(
        [
            X_test_fold,
            test_te,
            test_freq
        ],
        axis=1
    )

    # --------------------------------
    # Align categorical dtypes
    # --------------------------------
    categorical_cols = (
        X_train_fold
        .select_dtypes(
            include=["object", "str", "category"]
        )
        .columns
        .tolist()
    )

    for col in categorical_cols:

        categories = pd.concat(
            [
                X_train_fold[col].astype(str),
                X_valid_fold[col].astype(str),
                X_test_fold[col].astype(str)
            ],
            ignore_index=True
        ).unique().tolist()

        X_train_fold[col] = pd.Categorical(
            X_train_fold[col].astype(str),
            categories=categories
        )

        X_valid_fold[col] = pd.Categorical(
            X_valid_fold[col].astype(str),
            categories=categories
        )

        X_test_fold[col] = pd.Categorical(
            X_test_fold[col].astype(str),
            categories=categories
        )

    # --------------------------------
    # Model
    # --------------------------------
    model = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42 + fold,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_fold,
        y_outer_train.reset_index(drop=True),

        eval_X=X_valid_fold,
        eval_y=y_outer_valid.reset_index(drop=True),

        eval_metric="auc",

        callbacks=[
            early_stopping(
                150,
                first_metric_only=True
            ),
            log_evaluation(0)
        ]
    )

    valid_pred = model.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fold
    )[:, 1]

    oof_predictions_artifact[
        valid_idx
    ] = valid_pred

    test_predictions_artifact += (
        test_pred / 5
    )

    fold_auc = roc_auc_score(
        y_outer_valid,
        valid_pred
    )

    fold_scores_artifact.append(
        fold_auc
    )

    best_iterations_artifact.append(
        model.best_iteration_
    )

    print(
        f"Fold {fold} AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )


OUTER FOLD 1
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[842]	valid_0's auc: 0.94459	valid_0's binary_logloss: 0.22111
Evaluated only: auc
Fold 1 AUC: 0.944590
Best iteration: 842

OUTER FOLD 2
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[858]	valid_0's auc: 0.945256	valid_0's binary_logloss: 0.219798
Evaluated only: auc
Fold 2 AUC: 0.945256
Best iteration: 858

OUTER FOLD 3
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[938]	valid_0's auc: 0.946335	valid_0's binary_logloss: 0.217954
Evaluated only: auc
Fold 3 AUC: 0.946335
Best iteration: 938

OUTER FOLD 4
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[649]	valid_0's auc: 0.945939	valid_0's binary_logloss: 0.218623
Evaluated only: auc
Fold 4 AUC: 0.945939
Best iteration: 649

OUTER FOLD 5
Training until validation scores don't i

In [34]:
artifact_oof_auc = roc_auc_score(
    y,
    oof_predictions_artifact
)

print("\nFold scores:")

for fold, score in enumerate(
    fold_scores_artifact,
    start=1
):
    print(
        f"Fold {fold}: "
        f"{score:.6f}"
    )

print()

print(
    f"Mean fold AUC: "
    f"{np.mean(fold_scores_artifact):.6f}"
)

print(
    f"OOF AUC:       "
    f"{artifact_oof_auc:.6f}"
)

print(
    f"Std AUC:       "
    f"{np.std(fold_scores_artifact):.6f}"
)

print()

print(
    "Best iterations:",
    best_iterations_artifact
)


Fold scores:
Fold 1: 0.944590
Fold 2: 0.945256
Fold 3: 0.946335
Fold 4: 0.945939
Fold 5: 0.945468

Mean fold AUC: 0.945518
OOF AUC:       0.945511
Std AUC:       0.000596

Best iterations: [842, 858, 938, 649, 783]


In [35]:
artifact_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "artifact_lgb_oof":
        oof_predictions_artifact
})

artifact_oof_df.to_csv(
    "../outputs/predictions/"
    "lightgbm_income_artifact_oof.csv",
    index=False
)

In [36]:
artifact_test_df = pd.DataFrame({
    "id": test["id"],
    "artifact_lgb_prediction":
        test_predictions_artifact
})

artifact_test_df.to_csv(
    "../outputs/predictions/"
    "lightgbm_income_artifact_test.csv",
    index=False
)

In [37]:
cat_oof = pd.read_csv(
    "../outputs/predictions/catboost_oof.csv"
)

xgb_oof = pd.read_csv(
    "../outputs/predictions/xgboost_oof.csv"
)

fe_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_feature_engineered_oof.csv"
)

artifact_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_income_artifact_oof.csv"
)

In [38]:
assert cat_oof["id"].equals(xgb_oof["id"])
assert cat_oof["id"].equals(fe_oof["id"])
assert cat_oof["id"].equals(artifact_oof["id"])

print("OOF files aligned.")

OOF files aligned.


In [39]:
corr_matrix = pd.DataFrame({
    "CatBoost": cat_oof["catboost_oof"],
    "XGBoost": xgb_oof["xgboost_oof"],
    "FE_LightGBM": fe_oof["lgb_fe_oof"],
    "Artifact_LightGBM": artifact_oof["artifact_lgb_oof"]
}).corr()

corr_matrix

,CatBoost,XGBoost,FE_LightGBM,Artifact_LightGBM
CatBoost,1.000000,0.997973,0.997185,0.979795
XGBoost,0.997973,1.000000,0.997179,0.980184
FE_LightGBM,0.997185,0.997179,1.000000,0.981238
Artifact_LightGBM,0.979795,0.980184,0.981238,1.000000


In [40]:
y_ensemble = artifact_oof["target"]

cat_rank = cat_oof["catboost_oof"].rank(
    method="average",
    pct=True
)

xgb_rank = xgb_oof["xgboost_oof"].rank(
    method="average",
    pct=True
)

fe_rank = fe_oof["lgb_fe_oof"].rank(
    method="average",
    pct=True
)

artifact_rank = artifact_oof["artifact_lgb_oof"].rank(
    method="average",
    pct=True
)

In [41]:
old_ensemble_rank = (
    0.20 * cat_rank
    + 0.35 * fe_rank
    + 0.45 * xgb_rank
)

old_ensemble_auc = roc_auc_score(
    y_ensemble,
    old_ensemble_rank
)

old_ensemble_auc

0.9420459541460551

In [42]:
artifact_blend_results = []

for artifact_w in np.arange(0.0, 1.01, 0.05):

    old_w = 1.0 - artifact_w

    blended = (
        artifact_w * artifact_rank
        + old_w * old_ensemble_rank
    )

    auc = roc_auc_score(
        y_ensemble,
        blended
    )

    artifact_blend_results.append({
        "Artifact_Weight": artifact_w,
        "Old_Ensemble_Weight": old_w,
        "OOF_AUC": auc
    })

artifact_blend_results = pd.DataFrame(
    artifact_blend_results
)

artifact_blend_results.sort_values(
    "OOF_AUC",
    ascending=False
)

,Artifact_Weight,Old_Ensemble_Weight,OOF_AUC
18,0.90,0.10,0.945539
19,0.95,0.05,0.945536
17,0.85,0.15,0.945522
20,1.00,0.00,0.945511
16,0.80,0.20,0.945486
15,0.75,0.25,0.945429
14,0.70,0.30,0.945353
13,0.65,0.35,0.945256
12,0.60,0.40,0.945139
11,0.55,0.45,0.945001


In [43]:
best_artifact_blend = (
    artifact_blend_results
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .iloc[0]
)

best_artifact_blend

Artifact_Weight        0.900000
Old_Ensemble_Weight    0.100000
OOF_AUC                0.945539
Name: 18, dtype: float64

In [44]:
pair_results = []

other_models = {
    "CatBoost": cat_rank,
    "XGBoost": xgb_rank,
    "FE_LightGBM": fe_rank
}

for model_name, model_rank in other_models.items():

    for artifact_w in np.arange(
        0.50,
        1.01,
        0.05
    ):

        other_w = 1 - artifact_w

        blended = (
            artifact_w * artifact_rank
            + other_w * model_rank
        )

        auc = roc_auc_score(
            y_ensemble,
            blended
        )

        pair_results.append({
            "Other_Model": model_name,
            "Artifact_Weight": artifact_w,
            "Other_Weight": other_w,
            "OOF_AUC": auc
        })

pair_results = pd.DataFrame(pair_results)

pair_results.sort_values(
    "OOF_AUC",
    ascending=False
).head(15)

,Other_Model,Artifact_Weight,Other_Weight,OOF_AUC
19,XGBoost,0.90,1.000000e-01,0.945544
8,CatBoost,0.90,1.000000e-01,0.945540
20,XGBoost,0.95,5.000000e-02,0.945539
9,CatBoost,0.95,5.000000e-02,0.945537
31,FE_LightGBM,0.95,5.000000e-02,0.945529
18,XGBoost,0.85,1.500000e-01,0.945528
30,FE_LightGBM,0.90,1.000000e-01,0.945525
7,CatBoost,0.85,1.500000e-01,0.945521
10,CatBoost,1.00,-4.440892e-16,0.945511
32,FE_LightGBM,1.00,-4.440892e-16,0.945511


In [45]:
print(f"Artifact model alone:    {artifact_oof_auc:.6f}")
print(f"Old ensemble:            {old_ensemble_auc:.6f}")

print()
print("Best artifact + old ensemble:")
print(best_artifact_blend)

print()
print("Best artifact pair blends:")
print(
    pair_results
    .sort_values("OOF_AUC", ascending=False)
    .head(10)
)

Artifact model alone:    0.945511
Old ensemble:            0.942046

Best artifact + old ensemble:
Artifact_Weight        0.900000
Old_Ensemble_Weight    0.100000
OOF_AUC                0.945539
Name: 18, dtype: float64

Best artifact pair blends:
    Other_Model  Artifact_Weight  Other_Weight   OOF_AUC
19      XGBoost             0.90  1.000000e-01  0.945544
8      CatBoost             0.90  1.000000e-01  0.945540
20      XGBoost             0.95  5.000000e-02  0.945539
9      CatBoost             0.95  5.000000e-02  0.945537
31  FE_LightGBM             0.95  5.000000e-02  0.945529
18      XGBoost             0.85  1.500000e-01  0.945528
30  FE_LightGBM             0.90  1.000000e-01  0.945525
7      CatBoost             0.85  1.500000e-01  0.945521
10     CatBoost             1.00 -4.440892e-16  0.945511
32  FE_LightGBM             1.00 -4.440892e-16  0.945511


In [46]:
def make_artifact_keys_v2(df):
    keys = pd.DataFrame(index=df.index)

    income = df["Annual_Income_USD"]
    commute = df["Daily_Commute_km"]

    # -------------------------
    # Income artifacts
    # -------------------------
    keys["Income_Exact"] = (
        income
        .round(0)
        .astype("Int64")
        .astype(str)
    )

    keys["Income_Floor_50"] = (
        np.floor(income / 50) * 50
    ).astype("Int64").astype(str)

    keys["Income_Floor_250"] = (
        np.floor(income / 250) * 250
    ).astype("Int64").astype(str)

    keys["Income_Floor_500"] = (
        np.floor(income / 500) * 500
    ).astype("Int64").astype(str)

    # -------------------------
    # Commute artifacts
    # -------------------------

    # Original values have roughly 0.1 km precision
    keys["Commute_Exact"] = (
        (commute * 10)
        .round()
        .astype("Int64")
        .astype(str)
    )

    keys["Commute_Floor_1"] = (
        np.floor(commute)
    ).astype("Int64").astype(str)

    keys["Commute_Floor_5"] = (
        np.floor(commute / 5) * 5
    ).astype("Int64").astype(str)

    keys["Commute_Floor_10"] = (
        np.floor(commute / 10) * 10
    ).astype("Int64").astype(str)

    return keys

In [47]:
artifact_keys_v2 = make_artifact_keys_v2(train)

artifact_keys_v2.nunique()

Income_Exact        13214
Income_Floor_50      2026
Income_Floor_250      533
Income_Floor_500      284
Commute_Exact         805
Commute_Floor_1        89
Commute_Floor_5        19
Commute_Floor_10       10
dtype: int64

In [48]:
def create_artifact_features_v2(
    train_df,
    valid_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    train_keys = make_artifact_keys_v2(train_df)
    valid_keys = make_artifact_keys_v2(valid_df)

    train_te = pd.DataFrame(index=train_df.index)
    valid_te = pd.DataFrame(index=valid_df.index)

    train_freq = pd.DataFrame(index=train_df.index)
    valid_freq = pd.DataFrame(index=valid_df.index)

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        # -------------------------
        # Leak-free target encoding
        # -------------------------
        oof_values = np.zeros(len(train_df))

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):
            mapping, global_mean = build_smoothed_mapping(
                train_keys[col].iloc[inner_train_idx],
                y_train.iloc[inner_train_idx],
                smoothing=smoothing
            )

            oof_values[inner_valid_idx] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_te[f"TE_{col}"] = oof_values

        full_mapping, full_global_mean = build_smoothed_mapping(
            train_keys[col],
            y_train,
            smoothing=smoothing
        )

        valid_te[f"TE_{col}"] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        # -------------------------
        # Frequency encoding
        # -------------------------
        counts = train_keys[col].value_counts()

        train_freq[f"FREQ_{col}"] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[f"FREQ_{col}"] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_te,
        valid_te,
        train_freq,
        valid_freq
    )

In [49]:
probe_train_idx, probe_valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

probe_train = train.iloc[
    probe_train_idx
].copy()

probe_valid = train.iloc[
    probe_valid_idx
].copy()

y_probe_train = y.iloc[
    probe_train_idx
].copy()

y_probe_valid = y.iloc[
    probe_valid_idx
].copy()

In [50]:
(
    probe_train_te_v2,
    probe_valid_te_v2,
    probe_train_freq_v2,
    probe_valid_freq_v2
) = create_artifact_features_v2(
    probe_train,
    probe_valid,
    y_probe_train,
    smoothing=50,
    n_splits=5,
    random_state=42
)

In [51]:
probe_train_te_v2.shape, probe_valid_te_v2.shape

((534932, 8), (133733, 8))

In [52]:
probe_train_fe = add_features(
    probe_train
)

probe_valid_fe = add_features(
    probe_valid
)

In [53]:
X_probe_train_v2 = (
    probe_train_fe
    .drop(columns=["id", "Will_Buy_EV"])
    .reset_index(drop=True)
)

X_probe_valid_v2 = (
    probe_valid_fe
    .drop(columns=["id", "Will_Buy_EV"])
    .reset_index(drop=True)
)

In [54]:
X_probe_train_v2 = pd.concat(
    [
        X_probe_train_v2,
        probe_train_te_v2,
        probe_train_freq_v2
    ],
    axis=1
)

X_probe_valid_v2 = pd.concat(
    [
        X_probe_valid_v2,
        probe_valid_te_v2,
        probe_valid_freq_v2
    ],
    axis=1
)

In [55]:
X_probe_train_v2.shape, X_probe_valid_v2.shape

((534932, 40), (133733, 40))

In [56]:
categorical_cols_v2 = (
    X_probe_train_v2
    .select_dtypes(
        include=["object", "str", "category"]
    )
    .columns
    .tolist()
)

for col in categorical_cols_v2:

    categories = pd.concat(
        [
            X_probe_train_v2[col].astype(str),
            X_probe_valid_v2[col].astype(str)
        ],
        ignore_index=True
    ).unique().tolist()

    X_probe_train_v2[col] = pd.Categorical(
        X_probe_train_v2[col].astype(str),
        categories=categories
    )

    X_probe_valid_v2[col] = pd.Categorical(
        X_probe_valid_v2[col].astype(str),
        categories=categories
    )

In [57]:
artifact_v2_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [58]:
artifact_v2_model.fit(
    X_probe_train_v2,
    y_probe_train.reset_index(drop=True),

    eval_X=X_probe_valid_v2,
    eval_y=y_probe_valid.reset_index(drop=True),

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.945018	valid_0's binary_logloss: 0.220552
[400]	valid_0's auc: 0.945316	valid_0's binary_logloss: 0.219882
[600]	valid_0's auc: 0.945374	valid_0's binary_logloss: 0.219756
[800]	valid_0's auc: 0.945378	valid_0's binary_logloss: 0.21974
[1000]	valid_0's auc: 0.94537	valid_0's binary_logloss: 0.219738
Early stopping, best iteration is:
[881]	valid_0's auc: 0.94539	valid_0's binary_logloss: 0.219717
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [59]:
artifact_v2_pred = (
    artifact_v2_model
    .predict_proba(
        X_probe_valid_v2
    )[:, 1]
)

artifact_v2_auc = roc_auc_score(
    y_probe_valid,
    artifact_v2_pred
)

In [60]:
print(f"FE only:                 {0.941785:.6f}")
print(f"Income artifacts V1:     {0.945266:.6f}")
print(f"Income + commute V2:     {artifact_v2_auc:.6f}")

print()

print(
    f"Gain vs V1: "
    f"{artifact_v2_auc - 0.945266:+.6f}"
)

print(
    f"Best iteration: "
    f"{artifact_v2_model.best_iteration_}"
)

FE only:                 0.941785
Income artifacts V1:     0.945266
Income + commute V2:     0.945390

Gain vs V1: +0.000124
Best iteration: 881


In [61]:
artifact_v2_importance = pd.DataFrame({
    "Feature": X_probe_train_v2.columns,
    "Importance": artifact_v2_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

artifact_v2_importance[
    artifact_v2_importance[
        "Feature"
    ].str.contains(
        "Commute",
        regex=False
    )
]

,Feature,Importance
28,TE_Commute_Exact,1282
29,TE_Commute_Floor_1,1257
36,FREQ_Commute_Exact,853
17,Commute_x_Concern,656
2,Daily_Commute_km,519
37,FREQ_Commute_Floor_1,506
30,TE_Commute_Floor_5,506
23,HomeCharge_x_Commute,328
31,TE_Commute_Floor_10,320
38,FREQ_Commute_Floor_5,114


In [63]:
artifact_test = pd.read_csv(
    "../outputs/predictions/lightgbm_income_artifact_test.csv"
)

artifact_test.head()

,id,artifact_lgb_prediction
0,668665,0.028643
1,668666,0.020016
2,668667,0.004196
3,668668,0.002410
4,668669,0.028581


In [64]:
submission_005 = pd.DataFrame({
    "id": artifact_test["id"],
    "Will_Buy_EV": artifact_test["artifact_lgb_prediction"]
})

In [65]:
submission_005 = pd.DataFrame({
    "id": artifact_test["id"],
    "Will_Buy_EV": artifact_test["artifact_lgb_prediction"]
})

In [66]:
submission_005.shape

(286571, 2)

In [67]:
submission_005.isna().sum()

id             0
Will_Buy_EV    0
dtype: int64

In [68]:
submission_005["id"].equals(test["id"])

True

In [69]:
submission_005.to_csv(
    "../outputs/submissions/submission_005_income_artifact_lgbm.csv",
    index=False
)